# LightGBM on De-duplicated Dataset

This notebook trains a LightGBM classifier on the **de-duplicated dataset** with:

1. **Near-duplicate removal**: Articles with Jaccard similarity >= 0.7 grouped and deduplicated
2. **Enhanced source cleaning**: Additional patterns like "Reuters \n" removed
3. **Optuna-optimized hyperparameters**: Best parameters from hyperparameter tuning
4. **TF-IDF features**: 5000 features with unigrams + bigrams

**Goal**: Evaluate model performance on cleaner data without near-duplicate leakage.


In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings

# LightGBM
import lightgbm as lgb
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.inspection import permutation_importance

# Add project root to path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

warnings.filterwarnings('ignore', category=UserWarning)

print("✓ Imports successful")


✓ Imports successful


## 1. Load De-duplicated Data

Load the pre-processed de-duplicated dataset:
- `cleaned_dataset_deduped.csv`: Near-duplicates removed (Jaccard >= 0.7)
- `train_deduped.csv` / `test_deduped.csv`: 80/20 stratified split
- `features_train_tfidf_deduped.npz`: TF-IDF features


In [2]:
# Define paths
processed_dir = project_root / "data" / "processed"

# Load TF-IDF features
print("=" * 60)
print("LOADING DE-DUPLICATED DATA")
print("=" * 60)

print("\n1. Loading TF-IDF features...")
train_data = np.load(str(processed_dir / "features_train_tfidf_deduped.npz"), allow_pickle=True)
test_data = np.load(str(processed_dir / "features_test_tfidf_deduped.npz"), allow_pickle=True)

X_train = train_data["X"].item()  # Extract sparse matrix
y_train = train_data["y"]
X_test = test_data["X"].item()
y_test = test_data["y"]

print(f"   X_train shape: {X_train.shape}")
print(f"   X_test shape:  {X_test.shape}")

print(f"\n2. Label distribution:")
print(f"   Train: 0={np.sum(y_train==0)}, 1={np.sum(y_train==1)}")
print(f"   Test:  0={np.sum(y_test==0)}, 1={np.sum(y_test==1)}")

# Load vectorizer for feature names
print("\n3. Loading TF-IDF vectorizer...")
vectorizer = joblib.load(str(processed_dir / "tfidf_vectorizer_deduped.pkl"))
feature_names = vectorizer.get_feature_names_out()
print(f"   Vocabulary size: {len(feature_names)}")


LOADING DE-DUPLICATED DATA

1. Loading TF-IDF features...
   X_train shape: (34672, 5000)
   X_test shape:  (8668, 5000)

2. Label distribution:
   Train: 0=23394, 1=11278
   Test:  0=5848, 1=2820

3. Loading TF-IDF vectorizer...
   Vocabulary size: 5000


In [3]:
# Compare with original dataset
print("\n" + "=" * 60)
print("DATASET COMPARISON")
print("=" * 60)

# Load original features for comparison
orig_train = np.load(str(processed_dir / "features_train_tfidf.npz"), allow_pickle=True)
orig_test = np.load(str(processed_dir / "features_test_tfidf.npz"), allow_pickle=True)

print(f"\n{'Dataset':<20} {'Train':<15} {'Test':<15} {'Total':<15}")
print("-" * 65)
print(f"{'Original':<20} {orig_train['y'].shape[0]:<15} {orig_test['y'].shape[0]:<15} {orig_train['y'].shape[0] + orig_test['y'].shape[0]:<15}")
print(f"{'De-duplicated':<20} {X_train.shape[0]:<15} {X_test.shape[0]:<15} {X_train.shape[0] + X_test.shape[0]:<15}")
print(f"{'Removed':<20} {orig_train['y'].shape[0] - X_train.shape[0]:<15} {orig_test['y'].shape[0] - X_test.shape[0]:<15} {(orig_train['y'].shape[0] + orig_test['y'].shape[0]) - (X_train.shape[0] + X_test.shape[0]):<15}")



DATASET COMPARISON

Dataset              Train           Test            Total          
-----------------------------------------------------------------
Original             35551           8888            44439          
De-duplicated        34672           8668            43340          
Removed              879             220             1099           


## 2. Model Training with Optuna-Optimized Hyperparameters

Use the best hyperparameters found from Optuna optimization:
- Best CV F1-score: 0.9331
- 100 trials with TPE sampler


In [4]:
# Optuna-optimized hyperparameters
OPTUNA_PARAMS = {
    'n_estimators': 499,
    'learning_rate': 0.287572,
    'num_leaves': 46,
    'max_depth': 12,
    'min_child_samples': 59,
    'subsample': 0.631136,
    'colsample_bytree': 0.505511,
    'reg_alpha': 0.003473,
    'reg_lambda': 0.003938,
    'objective': 'binary',
    'verbosity': -1,
    'random_state': 42,
    'n_jobs': -1
}

print("=" * 60)
print("OPTUNA-OPTIMIZED HYPERPARAMETERS")
print("=" * 60)
print("\nBest parameters from Optuna (100 trials, TPE sampler):")
for param, value in OPTUNA_PARAMS.items():
    if isinstance(value, float):
        print(f"  {param}: {value:.6f}")
    else:
        print(f"  {param}: {value}")


OPTUNA-OPTIMIZED HYPERPARAMETERS

Best parameters from Optuna (100 trials, TPE sampler):
  n_estimators: 499
  learning_rate: 0.287572
  num_leaves: 46
  max_depth: 12
  min_child_samples: 59
  subsample: 0.631136
  colsample_bytree: 0.505511
  reg_alpha: 0.003473
  reg_lambda: 0.003938
  objective: binary
  verbosity: -1
  random_state: 42
  n_jobs: -1


In [ ]:
print("\n" + "=" * 60)
print("TRAINING LGBM MODEL")
print("=" * 60)

# Create and train model
print("\n1. Initializing LightGBM with Optuna parameters...")
model = lgb.LGBMClassifier(**OPTUNA_PARAMS)

print("\n2. Training model...")
model.fit(X_train, y_train)
print(f"   ✓ Model trained on {X_train.shape[0]:,} samples")

# Make predictions
print("\n3. Making predictions on test set...")
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]
print("   ✓ Predictions complete")



TRAINING LGBM MODEL

1. Initializing LightGBM with Optuna parameters...

2. Training model...


/Users/alanye/miniconda3/envs/fake-news-detection/lib/python3.10/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


## 3. Model Evaluation


In [ ]:
# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average='macro')
report = classification_report(y_test, y_pred, target_names=['Real (0)', 'Fake (1)'], output_dict=True)
conf_matrix = confusion_matrix(y_test, y_pred)

print("=" * 60)
print("MODEL EVALUATION (De-duplicated Dataset)")
print("=" * 60)

print(f"\nAccuracy: {accuracy:.4f}")
print(f"F1-score (macro): {f1_macro:.4f}")

print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Real (0)', 'Fake (1)']))


In [ ]:
# Confusion Matrix visualization
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Purples', ax=ax,
            xticklabels=['Real (0)', 'Fake (1)'], yticklabels=['Real (0)', 'Fake (1)'])
ax.set_title(f"LightGBM (De-duplicated + Optuna) - Accuracy: {accuracy:.4f}")
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')
plt.tight_layout()
plt.show()


In [ ]:
# Compare with original dataset results
print("=" * 60)
print("COMPARISON: Original vs De-duplicated Dataset")
print("=" * 60)

# Original dataset results (from lgbm_optuna_experiment.ipynb)
original_accuracy = 0.9412
original_f1_macro = 0.9336

print(f"\n{'Metric':<20} {'Original':<15} {'De-duplicated':<15} {'Difference':<15}")
print("-" * 65)
print(f"{'Accuracy':<20} {original_accuracy:.4f}{'':>9} {accuracy:.4f}{'':>9} {accuracy - original_accuracy:+.4f}")
print(f"{'F1-score (macro)':<20} {original_f1_macro:.4f}{'':>9} {f1_macro:.4f}{'':>9} {f1_macro - original_f1_macro:+.4f}")

print(f"\nNote: Lower performance on de-duplicated data may indicate")
print(f"the original model benefited from near-duplicate leakage.")


## 4. Feature Importance Analysis

Analyze which words/n-grams are most predictive using:
1. **MDI (Mean Decrease in Impurity)**: LGBM's built-in feature_importances_
2. **Permutation Feature Importance**: Measures accuracy drop when shuffling
3. **SHAP**: Game-theoretic feature attributions


In [ ]:
print("=" * 60)
print("1. MDI (Mean Decrease in Impurity) Feature Importance")
print("   (Built-in LGBM feature_importances_ based on gain)")
print("=" * 60)

# MDI Feature Importance
mdi_importances = model.feature_importances_
mdi_df = pd.DataFrame({
    'feature': feature_names,
    'importance': mdi_importances
}).sort_values('importance', ascending=False)

print(f"\nTop 20 features by MDI:")
print(mdi_df.head(20).to_string(index=False))


In [ ]:
print("\n" + "=" * 60)
print("2. Permutation Feature Importance")
print("   (Measures accuracy drop when shuffling features)")
print("=" * 60)

# Use a sample for permutation importance
perm_sample_size = min(2000, X_test.shape[0])
print(f"\nUsing {perm_sample_size} samples for permutation importance...")

# Convert sparse to dense for the sample
if hasattr(X_test, 'toarray'):
    X_test_sample = X_test[:perm_sample_size].toarray()
else:
    X_test_sample = X_test[:perm_sample_size]
y_test_sample = y_test[:perm_sample_size]

print("Computing permutation importance (this may take a moment)...")
perm_result = permutation_importance(
    model, X_test_sample, y_test_sample, 
    n_repeats=10, random_state=42, n_jobs=-1, scoring='accuracy'
)

perm_df = pd.DataFrame({
    'feature': feature_names,
    'importance_mean': perm_result.importances_mean,
    'importance_std': perm_result.importances_std
}).sort_values('importance_mean', ascending=False)

print(f"\nTop 20 features by Permutation Importance:")
print(perm_df.head(20).to_string(index=False))


In [ ]:
print("\n" + "=" * 60)
print("3. SHAP (SHapley Additive exPlanations)")
print("   (Game-theoretic feature attributions)")
print("=" * 60)

try:
    import shap
    
    # Use a sample of test data for SHAP
    sample_size = min(1000, X_test.shape[0])
    X_sample = X_test[:sample_size].toarray()
    
    print(f"\nComputing SHAP values on {sample_size} samples...")
    
    # Create SHAP explainer
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_sample)
    
    # For binary classification, take absolute mean of SHAP values
    if isinstance(shap_values, list):
        shap_importance = np.abs(shap_values[1]).mean(axis=0)
    else:
        shap_importance = np.abs(shap_values).mean(axis=0)
    
    shap_df = pd.DataFrame({
        'feature': feature_names,
        'shap_importance': shap_importance
    }).sort_values('shap_importance', ascending=False)
    
    print(f"\nTop 20 features by SHAP importance:")
    print(shap_df.head(20).to_string(index=False))
    
    shap_available = True
except ImportError:
    print("\nSHAP library not installed. Install with: pip install shap")
    shap_available = False
except Exception as e:
    print(f"\nError computing SHAP values: {e}")
    shap_available = False


In [ ]:
# Visualize feature importance comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 8))

# MDI Feature Importance
ax1 = axes[0]
top_mdi = mdi_df.head(20)
y_pos = np.arange(len(top_mdi))
ax1.barh(y_pos, top_mdi['importance'].values, color='steelblue', alpha=0.8)
ax1.set_yticks(y_pos)
ax1.set_yticklabels(top_mdi['feature'].values)
ax1.invert_yaxis()
ax1.set_xlabel('Importance (Gain)')
ax1.set_title('MDI Feature Importance\n(Built-in LGBM)', fontweight='bold')
ax1.grid(axis='x', alpha=0.3)

# Permutation Feature Importance
ax2 = axes[1]
top_perm = perm_df.head(20)
y_pos = np.arange(len(top_perm))
ax2.barh(y_pos, top_perm['importance_mean'].values, color='crimson', alpha=0.8, 
         xerr=top_perm['importance_std'].values, capsize=3)
ax2.set_yticks(y_pos)
ax2.set_yticklabels(top_perm['feature'].values)
ax2.invert_yaxis()
ax2.set_xlabel('Mean Accuracy Decrease')
ax2.set_title('Permutation Feature Importance\n(Accuracy Drop)', fontweight='bold')
ax2.grid(axis='x', alpha=0.3)

# SHAP Feature Importance
ax3 = axes[2]
if shap_available:
    top_shap = shap_df.head(20)
    y_pos = np.arange(len(top_shap))
    ax3.barh(y_pos, top_shap['shap_importance'].values, color='forestgreen', alpha=0.8)
    ax3.set_yticks(y_pos)
    ax3.set_yticklabels(top_shap['feature'].values)
    ax3.invert_yaxis()
    ax3.set_xlabel('Mean |SHAP Value|')
    ax3.set_title('SHAP Feature Importance\n(Shapley Values)', fontweight='bold')
    ax3.grid(axis='x', alpha=0.3)
else:
    ax3.text(0.5, 0.5, 'SHAP not available', ha='center', va='center', fontsize=14)
    ax3.set_title('SHAP Feature Importance', fontweight='bold')

plt.suptitle('Feature Importance Analysis (De-duplicated Dataset)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(str(project_root / "results" / "figures" / "lgbm_deduped_feature_importance.png"), dpi=150, bbox_inches='tight')
plt.show()
print(f"\nFigure saved to: results/figures/lgbm_deduped_feature_importance.png")


In [ ]:
# Feature importance overlap analysis
print("=" * 60)
print("FEATURE IMPORTANCE METHODS COMPARISON")
print("=" * 60)

# Get top 20 features from each method
mdi_top20 = set(mdi_df.head(20)['feature'].values)
perm_top20 = set(perm_df.head(20)['feature'].values)

if shap_available:
    shap_top20 = set(shap_df.head(20)['feature'].values)
    
    # Find overlapping features
    all_methods_overlap = mdi_top20 & perm_top20 & shap_top20
    mdi_perm_overlap = mdi_top20 & perm_top20
    mdi_shap_overlap = mdi_top20 & shap_top20
    perm_shap_overlap = perm_top20 & shap_top20
    
    print(f"\nFeatures in Top 20 across ALL methods ({len(all_methods_overlap)}/20):")
    print(sorted(all_methods_overlap))
    print(f"\nMDI & Permutation overlap: {len(mdi_perm_overlap)}/20")
    print(f"MDI & SHAP overlap: {len(mdi_shap_overlap)}/20")
    print(f"Permutation & SHAP overlap: {len(perm_shap_overlap)}/20")
else:
    mdi_perm_overlap = mdi_top20 & perm_top20
    print(f"\nMDI & Permutation overlap ({len(mdi_perm_overlap)}/20):")
    print(sorted(mdi_perm_overlap))


## 5. Save Model


In [ ]:
# Save model
model_path = project_root / "results" / "models" / "lgbm_deduped.pkl"
joblib.dump(model, str(model_path))
print(f"✓ Model saved to: {model_path}")


## 6. Summary


In [ ]:
print("=" * 60)
print("LGBM DE-DUPLICATED EXPERIMENT SUMMARY")
print("=" * 60)

print(f"\n** Dataset **")
print(f"  Original articles: 44,439")
print(f"  After de-duplication: {X_train.shape[0] + X_test.shape[0]:,}")
print(f"  Near-duplicates removed: {44439 - (X_train.shape[0] + X_test.shape[0]):,}")
print(f"  Train samples: {X_train.shape[0]:,}")
print(f"  Test samples: {X_test.shape[0]:,}")

print(f"\n** Model Configuration **")
print(f"  Model: LightGBM Classifier")
print(f"  Features: TF-IDF ({len(feature_names)} features)")
print(f"  Hyperparameters: Optuna-optimized (100 trials)")

print(f"\n** Performance **")
print(f"  Accuracy: {accuracy:.4f}")
print(f"  F1-score (macro): {f1_macro:.4f}")

print(f"\n** Classification Report **")
for label, name in [('Real (0)', 'Real'), ('Fake (1)', 'Fake')]:
    print(f"  {name}:")
    print(f"    Precision: {report[label]['precision']:.4f}")
    print(f"    Recall:    {report[label]['recall']:.4f}")
    print(f"    F1-score:  {report[label]['f1-score']:.4f}")

print(f"\n** Top 10 Features by MDI **")
for i, (_, row) in enumerate(mdi_df.head(10).iterrows(), 1):
    print(f"  {i}. {row['feature']}: {row['importance']:.0f}")

print(f"\n** Key Observations **")
diff = accuracy - original_accuracy
if diff < -0.01:
    print(f"  - Accuracy dropped by {abs(diff):.4f} after de-duplication")
    print(f"  - This suggests near-duplicate leakage inflated original results")
elif diff > 0.01:
    print(f"  - Accuracy improved by {diff:.4f} after de-duplication")
    print(f"  - Cleaner data led to better generalization")
else:
    print(f"  - Accuracy similar to original ({diff:+.4f})")
    print(f"  - Near-duplicate removal had minimal impact on performance")
